# 01 — Ground truth and position calibration

Loads `Dense_Grid_A`, applies the InvOLS position calibration, and establishes the two facts every later result rests on: the ground-truth D-NS and the measured mode structure.

Paths come from `config.py` (override with `AMM_*` env vars).

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0,'..'); sys.path.insert(0,'../src')
from setup_data import load
x,f,Z,ires,ps = load('A','calibrated')
A=np.abs(Z)
print(f'{len(x)} positions x {len(f)} bins  |  resonance {f[ires]/1e3:.2f} kHz')
print('calibration:',ps)

The calibration is an affine map fitted from InvOLS (optical-lever sensitivity varies as the laser moves along the lever, so it is an independent position ruler). Scale 1.0244, offset −4.73 µm, rms 0.60 µm.

Note: for a Chebyshev spatial basis an affine remap of the axis is a **null operation** — calibration matters for physical interpretation (D-NS in µm), not for the low-rank baseline's accuracy.

In [ ]:
from activemodemap.lowrank import classify_null_from_map, resonance_index
cl=classify_null_from_map(x, A.astype(complex), f, resonance_index(f, A.astype(complex)))
print('ground-truth D-NS: %.2f um  (%s)'%(cl['x_null_um'], cl['status']))

In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
im=ax.pcolormesh(x, f/1e3, 20*np.log10(np.maximum(A,1e-16)/A.max()).T, vmin=-60, vmax=0, shading='nearest')
ax.axvline(cl['x_null_um'], color='m', ls='--'); ax.set_xlabel('position from clamp (um)'); ax.set_ylabel('kHz')
plt.colorbar(im,label='|Z| dB re max');

## The measured higher mode

The full 25–1775 kHz sweep contains flexural mode 2 at 902.41 kHz → **f₂/f₁ = 3.080** (between clamped–clamped 2.757 and clamped–pinned 3.240). This single number turns out to carry most of the contact-stiffness information — see notebook 02 and `docs/higher-modes-and-k1.md`.

In [ ]:
from config import DENSE_GRID_A
dn=np.load(str(DENSE_GRID_A/'DenseReference.npz'),allow_pickle=True)
ff,ZZ=dn['freq_Hz'],np.abs(dn['Z']).max(0)
from scipy.signal import find_peaks
pk,_=find_peaks(np.log(ZZ+1e-30),prominence=0.8,distance=40)
for i in pk: print('%8.2f kHz  %5.1f dB'%(ff[i]/1e3,20*np.log10(ZZ[i]/ZZ.max())))
print('f2/f1 =', ff[pk[1]]/ff[pk[0]])